# 2 — ASD prevalence ETL (OECD cohort)

## Objective

This notebook performs the ETL process to transform the raw
GBD ASD prevalence dataset into a clean and consistent analytical dataset.

Input datasets:

- `data/1_raw/ihme_gbd_asd_prevalence_oecd_1990_2023.csv`
- `data/3_processed/oecd_members.csv`

Main steps:

1. Load raw GBD dataset
2. Standardize column names
3. Normalize categorical variables
4. Validate dataset structure and cohort consistency
5. Export processed dataset

## 2.1 Environment and project setup

Prepare the notebook environment and project configuration.

Includes:
- core library imports
- environment validation
- project path setup

In [22]:
# Import core libraries and project paths for data access and ETL processing

import pandas as pd
import sys
from src.paths import RAW_DIR, PROCESSED_DIR, ensure_data_dirs

In [23]:
# Validate environment configuration and library availability for ETL execution

print("Python version:", sys.version.split()[0])

try:
    import src
    print("Project package import: OK")
except ImportError:
    print("ERROR: project package 'src' not found")

print("pandas version:", pd.__version__)

Python version: 3.13.5
Project package import: OK
pandas version: 2.2.3


In [24]:
# Ensure project data directories exist and validate path configuration for ETL workflow

ensure_data_dirs()

print("RAW_DIR:", RAW_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)

RAW_DIR: C:\Users\samai\OneDrive\Escritorio\PROYECTO AUTISMO\autism-diagnosis-gender-gap\data\1_raw
PROCESSED_DIR: C:\Users\samai\OneDrive\Escritorio\PROYECTO AUTISMO\autism-diagnosis-gender-gap\data\3_processed


## 2.2 Data ingestion

Load the raw ASD prevalence dataset from the project raw data directory.

Steps:
- locate the raw dataset file
- load the dataset into a pandas DataFrame

In [25]:
# Load raw ASD prevalence dataset and validate initial structure

raw_path = RAW_DIR / "ihme_gbd_asd_prevalence_oecd_1990_2023.csv"
assert raw_path.exists(), f"Raw dataset not found: {raw_path}"

df_raw = pd.read_csv(raw_path)

print("Shape:", df_raw.shape)
print("\nYear range:", df_raw["year"].min(), "-", df_raw["year"].max())
print("\nSex categories:", df_raw["sex_name"].unique())
print("\nAge categories:", df_raw["age_name"].unique())

# Inspect raw dataset sample
df_raw.head()

Shape: (38760, 18)

Year range: 1990 - 2023

Sex categories: ['Hombre' 'Mujer']

Age categories: ['Menores de 5 años' '5-9 años' '10-14 años' '15-19 años' '20-24 años'
 '25-29 años' '30-34 años' '35-39 años' '40-44 años' '45-49 años'
 '50-54 años' '55-59 años' '60-64 años' '65-69 años' '70+ años']


,population_group_id,population_group_name,measure_id,measure_name,location_id,location_name,sex_id,sex_name,age_id,age_name,cause_id,cause_name,metric_id,metric_name,year,val,upper,lower
0,1,Toda la población,5,Prevalencia,55,Eslovenia,1,Hombre,1,Menores de 5 años,575,Trastornos del espectro autista,3,Tasa,1991,939.037586,1855.692963,459.595189
1,1,Toda la población,5,Prevalencia,55,Eslovenia,2,Mujer,1,Menores de 5 años,575,Trastornos del espectro autista,3,Tasa,1991,363.753608,718.021074,177.685811
2,1,Toda la población,5,Prevalencia,55,Eslovenia,1,Hombre,6,5-9 años,575,Trastornos del espectro autista,3,Tasa,1991,920.239063,1984.293942,435.700531
3,1,Toda la población,5,Prevalencia,55,Eslovenia,2,Mujer,6,5-9 años,575,Trastornos del espectro autista,3,Tasa,1991,356.527957,772.392177,168.000119
4,1,Toda la población,5,Prevalencia,55,Eslovenia,1,Hombre,7,10-14 años,575,Trastornos del espectro autista,3,Tasa,1991,912.201931,1979.073395,358.173398


## 2.3 Data transformation

Transform the raw GBD dataset into a clean and consistent analytical dataset.

Steps:
- create a working copy of the raw dataset
- inspect dataset structure and categorical variables
- normalize categorical variables
- rename analytical measure columns
- prepare dataset structure for downstream analysis

In [26]:
# Create working dataset to preserve raw data integrity during ETL process

df = df_raw.copy()

In [27]:
# Generate structural summary of the dataset to support ETL inspection and validation

def dataset_summary(df):

    summary = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "n_unique": df.nunique(),
        "unique_values": [df[col].dropna().unique() for col in df.columns]
    })

    print("DATASET SHAPE")
    print("-------------")
    print(f"Rows: {df.shape[0]}")
    print(f"Columns: {df.shape[1]}\n")

    return summary

In [28]:
# Generate structural summary of the working dataset for ETL inspection

eda_summary = dataset_summary(df)
eda_summary

DATASET SHAPE
-------------
Rows: 38760
Columns: 18



,dtype,n_unique,unique_values
population_group_id,int64,1,[1]
population_group_name,object,1,[Toda la población]
measure_id,int64,1,[5]
measure_name,object,1,[Prevalencia]
location_id,int64,38,"[55, 155, 98, 125, 67, 89, 75, 90, 79, 81, 58,..."
location_name,object,38,"[Eslovenia, Turquía, Chile, Colombia, Japón, H..."
sex_id,int64,2,"[1, 2]"
sex_name,object,2,"[Hombre, Mujer]"
age_id,int64,15,"[1, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17..."
age_name,object,15,"[Menores de 5 años, 5-9 años, 10-14 años, 15-1..."


In [29]:
# Identify categorical variables to support normalization during ETL

categorical_summary = eda_summary[
    eda_summary["dtype"].isin(["object", "category"])
]

categorical_summary

,dtype,n_unique,unique_values
population_group_name,object,1,[Toda la población]
measure_name,object,1,[Prevalencia]
location_name,object,38,"[Eslovenia, Turquía, Chile, Colombia, Japón, H..."
sex_name,object,2,"[Hombre, Mujer]"
age_name,object,15,"[Menores de 5 años, 5-9 años, 10-14 años, 15-1..."
cause_name,object,1,[Trastornos del espectro autista]
metric_name,object,1,[Tasa]


In [30]:
# Identify key categorical variables requiring normalization (exclude single-value variables)

cat_key_columns = categorical_summary[
    categorical_summary["n_unique"] > 1
]

cat_key_columns

,dtype,n_unique,unique_values
location_name,object,38,"[Eslovenia, Turquía, Chile, Colombia, Japón, H..."
sex_name,object,2,"[Hombre, Mujer]"
age_name,object,15,"[Menores de 5 años, 5-9 años, 10-14 años, 15-1..."


In [31]:
# Inspect categorical values to support normalization and detect inconsistencies

for col in cat_key_columns.index:
    print(f"\nUnique values in {col}:")
    values = df[col].unique()
    print(sorted(values) if col in ["location_name", "sex_name"] else values)


Unique values in location_name:
['Alemania', 'Australia', 'Austria', 'Bélgica', 'Canadá', 'Chile', 'Colombia', 'Corea del Sur', 'Costa Rica', 'Dinamarca', 'Eslovaquia', 'Eslovenia', 'España', 'Estados Unidos de América', 'Estonia', 'Finlandia', 'Francia', 'Grecia', 'Holanda', 'Hungría', 'Irlanda', 'Islandia', 'Israel', 'Italia', 'Japón', 'Letonia', 'Lituania', 'Luxemburgo', 'México', 'Noruega', 'Nueva Zelanda', 'Polonia', 'Portugal', 'Reino Unido', 'República Checa', 'Suecia', 'Suiza', 'Turquía']

Unique values in sex_name:
['Hombre', 'Mujer']

Unique values in age_name:
['Menores de 5 años' '5-9 años' '10-14 años' '15-19 años' '20-24 años'
 '25-29 años' '30-34 años' '35-39 años' '40-44 años' '45-49 años'
 '50-54 años' '55-59 años' '60-64 años' '65-69 años' '70+ años']


In [32]:
# Define mappings to standardize categorical variables into consistent analytical labels

sex_mapping = {"Hombre": "Male", "Mujer": "Female"}

age_mapping = {
    "Menores de 5 años": "<5", "5-9 años": "5-9", "10-14 años": "10-14", "15-19 años": "15-19",
    "20-24 años": "20-24", "25-29 años": "25-29", "30-34 años": "30-34", "35-39 años": "35-39",
    "40-44 años": "40-44", "45-49 años": "45-49", "50-54 años": "50-54", "55-59 años": "55-59",
    "60-64 años": "60-64", "65-69 años": "65-69", "70+ años": "70+"
}

location_mapping = {
    "Alemania": "Germany", "Australia": "Australia", "Austria": "Austria", "Bélgica": "Belgium",
    "Canadá": "Canada", "Chile": "Chile", "Colombia": "Colombia", "Corea del Sur": "South Korea",
    "Costa Rica": "Costa Rica", "Dinamarca": "Denmark", "Eslovaquia": "Slovakia",
    "Eslovenia": "Slovenia", "España": "Spain", "Estados Unidos de América": "United States",
    "Estonia": "Estonia", "Finlandia": "Finland", "Francia": "France", "Grecia": "Greece",
    "Holanda": "Netherlands", "Hungría": "Hungary", "Irlanda": "Ireland", "Islandia": "Iceland",
    "Israel": "Israel", "Italia": "Italy", "Japón": "Japan", "Letonia": "Latvia",
    "Lituania": "Lithuania", "Luxemburgo": "Luxembourg", "México": "Mexico",
    "Noruega": "Norway", "Nueva Zelanda": "New Zealand", "Polonia": "Poland",
    "Portugal": "Portugal", "Reino Unido": "United Kingdom", "República Checa": "Czechia",
    "Suecia": "Sweden", "Suiza": "Switzerland", "Turquía": "Turkey"
}

In [33]:
# Apply categorical mappings and define ordered variables for consistent analysis

df["gender"] = df["sex_name"].map(sex_mapping)
df["age_range"] = df["age_name"].map(age_mapping)
df["country"] = df["location_name"].map(location_mapping)

# Define correct order for age ranges
age_order = [
    "<5","5-9","10-14","15-19","20-24","25-29","30-34",
    "35-39","40-44","45-49","50-54","55-59","60-64","65-69","70+"
]

df["age_range"] = pd.Categorical(df["age_range"], categories=age_order, ordered=True)

In [34]:
# Validate mapping completeness and consistency with OECD reference

oecd_reference = pd.read_csv(PROCESSED_DIR / "oecd_members.csv")

print("Missing values after mapping:")
print(df[["gender", "age_range", "country"]].isnull().sum())

missing_countries = set(df["country"].dropna().unique()) - set(oecd_reference["country"].unique())
assert len(missing_countries) == 0, f"Unmapped countries: {missing_countries}"

Missing values after mapping:
gender       0
age_range    0
country      0
dtype: int64


In [35]:
# Inspect normalized categorical variables after mapping

for col in ["country", "gender", "age_range"]:
    print(f"\nUnique values in {col}:")
    print(df[col].unique())


Unique values in country:
['Slovenia' 'Turkey' 'Chile' 'Colombia' 'Japan' 'Netherlands' 'Austria'
 'Norway' 'Finland' 'Germany' 'Estonia' 'Greece' 'Mexico' 'Hungary'
 'Australia' 'Spain' 'South Korea' 'Costa Rica' 'Slovakia' 'New Zealand'
 'Czechia' 'Canada' 'Italy' 'Switzerland' 'Latvia' 'Portugal'
 'United States' 'Belgium' 'Ireland' 'Sweden' 'Iceland' 'Poland'
 'United Kingdom' 'France' 'Israel' 'Lithuania' 'Denmark' 'Luxembourg']

Unique values in gender:
['Male' 'Female']

Unique values in age_range:
['<5', '5-9', '10-14', '15-19', '20-24', ..., '50-54', '55-59', '60-64', '65-69', '70+']
Length: 15
Categories (15, object): ['<5' < '5-9' < '10-14' < '15-19' ... '55-59' < '60-64' < '65-69' < '70+']


In [36]:
# Rename analytical measure columns for consistent and clear interpretation

df = df.rename(columns={"val": "prevalence", "upper": "upper_ui", "lower": "lower_ui"})

In [37]:
# Inspect renamed analytical columns to verify correct transformation

df[["prevalence", "upper_ui", "lower_ui"]].head()

,prevalence,upper_ui,lower_ui
0,939.037586,1855.692963,459.595189
1,363.753608,718.021074,177.685811
2,920.239063,1984.293942,435.700531
3,356.527957,772.392177,168.000119
4,912.201931,1979.073395,358.173398


In [38]:
# Build final analytical dataset with selected variables for downstream analysis

analytical_columns = ["country", "gender", "age_range", "year", "prevalence", "upper_ui", "lower_ui"]

df_processed = df[analytical_columns].copy()

In [39]:
# Validate structure of final analytical dataset before export

print("Shape:", df_processed.shape)

print("\nColumns:")
print(df_processed.columns)

# Inspect sample of processed dataset
df_processed.head()

Shape: (38760, 7)

Columns:
Index(['country', 'gender', 'age_range', 'year', 'prevalence', 'upper_ui',
       'lower_ui'],
      dtype='object')


,country,gender,age_range,year,prevalence,upper_ui,lower_ui
0,Slovenia,Male,<5,1991,939.037586,1855.692963,459.595189
1,Slovenia,Female,<5,1991,363.753608,718.021074,177.685811
2,Slovenia,Male,5-9,1991,920.239063,1984.293942,435.700531
3,Slovenia,Female,5-9,1991,356.527957,772.392177,168.000119
4,Slovenia,Male,10-14,1991,912.201931,1979.073395,358.173398


In [40]:
# Export processed analytical dataset for downstream analysis

output_file = PROCESSED_DIR / "gbd_asd_prevalence_oecd_processed.csv"

df_processed.to_csv(output_file, index=False)

print("Saved:", output_file)

Saved: C:\Users\samai\OneDrive\Escritorio\PROYECTO AUTISMO\autism-diagnosis-gender-gap\data\3_processed\gbd_asd_prevalence_oecd_processed.csv


In [41]:
# Validate exported dataset by reloading and inspecting structure

df_check = pd.read_csv(output_file)

print("Shape:", df_check.shape)

print("\nColumns:")
print(df_check.columns)

# Inspect sample of exported dataset
df_check.head()

Shape: (38760, 7)

Columns:
Index(['country', 'gender', 'age_range', 'year', 'prevalence', 'upper_ui',
       'lower_ui'],
      dtype='object')


,country,gender,age_range,year,prevalence,upper_ui,lower_ui
0,Slovenia,Male,<5,1991,939.037586,1855.692963,459.595189
1,Slovenia,Female,<5,1991,363.753608,718.021074,177.685811
2,Slovenia,Male,5-9,1991,920.239063,1984.293942,435.700531
3,Slovenia,Female,5-9,1991,356.527957,772.392177,168.000119
4,Slovenia,Male,10-14,1991,912.201931,1979.073395,358.173398
